# LeetCode 4112 — Maximize Score After Zeroing Subtrees (Medium)

Bài toán LeetCode:

```python
class Solution:
    def maxScore(self, parent: List[int], op: List[int], score: List[int]) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4112 (Weekly Contest 528 — câu 2).

Đây là bài **quy hoạch động trên cây (tree DP)** ở dạng dễ tiếp cận nhất: mỗi đỉnh đưa ra một quyết định về các con của nó, và các cây con **độc lập** với nhau. Ta sẽ học cách:

- đọc cây cho dưới dạng **mảng cha** (`parent`);
- đặt đúng **câu hỏi con** cho mỗi đỉnh: "cây con này giữ lại được nhiều điểm nhất là bao nhiêu?";
- phân biệt lựa chọn **bắt buộc** (`op = 1`) và **tùy chọn** (`op = 0`);
- cài đặt **không đệ quy** để chạy được với cây sâu `10^5`.

---

## Sau notebook này, bạn sẽ nắm được

1. Đề bài: thao tác "xoá về 0 một cây con", vì sao **thứ tự thao tác không quan trọng**.
2. Dựng danh sách con từ mảng `parent`, thứ tự BFS và cách duyệt ngược để xử lý con trước cha.
3. Brute Force thử mọi tổ hợp lựa chọn (`itertools.product`).
4. Công thức DP: `dfs(u) = tổng - min_con` (bắt buộc) hoặc `tổng - min(0, min_con)` (tùy chọn).
5. Cài đặt đệ quy (dễ hiểu) và lặp (an toàn), trace từng bước.
6. Lỗi hay gặp (kể cả bẫy `children[-1]` của Python), mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho một cây có gốc gồm `n` đỉnh `0..n-1`, gốc là `0`. Mảng `parent` cho biết cha của mỗi đỉnh (`parent[0] = -1`).

Mỗi đỉnh `i` có:

- `score[i]` — điểm của đỉnh (có thể âm),
- `op[i]` — loại thao tác (`0` hoặc `1`).

Chỉ **đỉnh không phải lá** (có ít nhất một con) mới có thao tác. **Mỗi đỉnh không phải lá phải được xử lý đúng một lần**, theo thứ tự tùy ý:

- `op[i] = 0`: chọn **nhiều nhất một** con của `i` (hoặc không chọn con nào), rồi **đặt về 0** điểm của **mọi đỉnh trong cây con** của con đó.
- `op[i] = 1`: chọn **đúng một** con của `i`, rồi đặt về 0 điểm của mọi đỉnh trong cây con của con đó.

Trả về **tổng điểm lớn nhất** có thể đạt được sau khi thực hiện mọi thao tác.

---

## Ví dụ 1

```python
parent = [-1, 0, 0, 0, 1, 2, 3, 3]
op     = [ 1, 1, 0, 0, 1, 1, 0, 1]
score  = [ 5, 6, 1, 2, 3, 2, 3, -3]
-> 16
```

```text
                 0 (op=1, 5)
           /          |          \
     1 (op=1, 6)  2 (op=0, 1)  3 (op=0, 2)
        |            |          /       \
     4 (3)        5 (2)      6 (3)     7 (-3)
```

- Các lá `4, 5, 6, 7` không có thao tác (dù `op` của chúng có giá trị, ta bỏ qua).
- Đỉnh `1` (`op = 1`) bắt buộc xoá một con → xoá con `4`.
- Đỉnh `2` (`op = 0`) không làm gì.
- Đỉnh `3` (`op = 0`) xoá con `7` (điểm âm, xoá đi là có lợi).
- Gốc `0` (`op = 1`) bắt buộc xoá một con → xoá con `2` (kéo theo đỉnh `5`).

Điểm cuối: `[5, 6, 0, 2, 0, 0, 3, 0]` → tổng `16`.

## Ví dụ 2

```python
parent = [-1, 0, 0], op = [0, 0, 0], score = [1, 2, 3]   ->   6
```

Chỉ gốc là đỉnh không phải lá, `op = 0` nên được phép không làm gì → giữ hết `1 + 2 + 3 = 6`.

## Ví dụ 3

```python
parent = [-1], op = [1], score = [-10]   ->   -10
```

Đỉnh duy nhất là lá → không có thao tác → tổng `-10`.

---

## Ràng buộc

```text
1 <= n <= 10^5
n == parent.length == op.length == score.length
parent[0] == -1, 0 <= parent[i] <= n - 1 với i >= 1
0 <= op[i] <= 1
-10^4 <= score[i] <= 10^4
parent luôn mô tả một cây hợp lệ có gốc 0
```

# 2. Kiến thức nền tảng

## 2.1. Từ mảng `parent` sang danh sách con

`parent[i] = p` nghĩa là `i` là con của `p`. Duyệt mọi `i` từ `1` (bỏ gốc!) và thêm `i` vào `children[p]`.

In [1]:
parent = [-1, 0, 0, 0, 1, 2, 3, 3]
n = len(parent)

children = [[] for _ in range(n)]
for i in range(1, n):                  # bắt đầu từ 1: gốc không có cha
    children[parent[i]].append(i)

for u in range(n):
    kind = "lá" if not children[u] else "không phải lá"
    print(f"đỉnh {u}: con = {children[u]} ({kind})")

đỉnh 0: con = [1, 2, 3] (không phải lá)
đỉnh 1: con = [4] (không phải lá)
đỉnh 2: con = [5] (không phải lá)
đỉnh 3: con = [6, 7] (không phải lá)
đỉnh 4: con = [] (lá)
đỉnh 5: con = [] (lá)
đỉnh 6: con = [] (lá)
đỉnh 7: con = [] (lá)


## 2.2. Thứ tự BFS bằng một list "vừa duyệt vừa nối thêm"

Mẹo gọn trong Python:

```python
order = [0]
for u in order:              # vòng for đọc cả các phần tử được nối thêm trong lúc lặp
    order.extend(children[u])
```

Kết quả là thứ tự **duyệt theo tầng (BFS)**: cha luôn đứng trước con. Duyệt `reversed(order)` thì **con luôn được xử lý trước cha** — đúng thứ tự cần cho DP từ lá lên gốc.

(Nối thêm vào list trong lúc đang duyệt bằng `for` là hợp lệ với `list` trong Python, vòng lặp sẽ đi tới cả các phần tử mới.)

In [2]:
order = [0]
for u in order:
    order.extend(children[u])

print("BFS order       :", order)
print("reversed (lá lên):", list(reversed(order)))

BFS order       : [0, 1, 2, 3, 4, 5, 6, 7]
reversed (lá lên): [7, 6, 5, 4, 3, 2, 1, 0]


## 2.3. Khởi động: tổng điểm của mỗi cây con

Bài DP cây đơn giản nhất: `subtree_sum[u] = score[u] + Σ subtree_sum[con]`. Tính theo `reversed(order)`.

In [3]:
score = [5, 6, 1, 2, 3, 2, 3, -3]

subtree_sum = [0] * n
for u in reversed(order):
    subtree_sum[u] = score[u] + sum(subtree_sum[c] for c in children[u])

print("subtree_sum =", subtree_sum)
print("tổng cả cây  =", subtree_sum[0], "=", sum(score))

subtree_sum = [19, 9, 3, 2, 3, 2, 3, -3]
tổng cả cây  = 19 = 19


## 2.4. `itertools.product` — tích Descartes

`product(A, B, C)` sinh mọi bộ `(a, b, c)` với `a ∈ A`, `b ∈ B`, `c ∈ C`. Dùng để liệt kê **mọi tổ hợp lựa chọn** trong cách vét cạn.

In [4]:
from itertools import product

for combo in product([4], [None, 5], ["x", "y"]):
    print(combo)

(4, None, 'x')
(4, None, 'y')
(4, 5, 'x')
(4, 5, 'y')


# 3. Hiểu kỹ thao tác: thứ tự có quan trọng không?

- "Đặt về 0" là thao tác **lũy đẳng**: đặt về 0 một đỉnh đã bằng 0 thì vẫn là 0.
- Lựa chọn của mỗi đỉnh (xoá con nào) **không phụ thuộc** vào thứ tự thực hiện.

Vậy kết quả cuối chỉ phụ thuộc vào **tập các lựa chọn**: một đỉnh bằng 0 khi và chỉ khi nó nằm trong cây con của **ít nhất một** con bị chọn xoá. Thứ tự thao tác không quan trọng.

Ta gọi một **kế hoạch** là bộ lựa chọn cho mọi đỉnh không phải lá:

- đỉnh `op = 1`: chọn một con trong danh sách con;
- đỉnh `op = 0`: chọn một con hoặc `None` (không xoá gì).

---

# 4. Cách 1 — Brute Force: thử mọi kế hoạch

In [5]:
from typing import List


def build_children(parent: List[int]) -> List[List[int]]:
    n = len(parent)
    children = [[] for _ in range(n)]
    for i in range(1, n):
        children[parent[i]].append(i)
    return children


def max_score_bruteforce(parent: List[int], op: List[int], score: List[int]) -> int:
    n = len(parent)
    children = build_children(parent)
    internal = [u for u in range(n) if children[u]]
    options = [children[u] + ([None] if op[u] == 0 else []) for u in internal]

    best = None
    for plan in product(*options):            # mỗi plan: lựa chọn của từng đỉnh trong
        zero = [False] * n
        for chosen in plan:
            if chosen is None:
                continue
            stack = [chosen]                  # đặt về 0 cả cây con của 'chosen'
            while stack:
                v = stack.pop()
                zero[v] = True
                stack.extend(children[v])
        total = sum(score[v] for v in range(n) if not zero[v])
        best = total if best is None else max(best, total)
    return best


print(max_score_bruteforce([-1, 0, 0, 0, 1, 2, 3, 3], [1, 1, 0, 0, 1, 1, 0, 1], [5, 6, 1, 2, 3, 2, 3, -3]))  # 16
print(max_score_bruteforce([-1, 0, 0], [0, 0, 0], [1, 2, 3]))   # 6
print(max_score_bruteforce([-1], [1], [-10]))                   # -10

16
6
-10


## Số kế hoạch tăng rất nhanh

Số kế hoạch là **tích** số lựa chọn của từng đỉnh không phải lá. Với một cây mà mỗi đỉnh trong có 2 con và `op = 0` (3 lựa chọn mỗi đỉnh), số kế hoạch là `3^(số đỉnh trong)`:

In [6]:
def count_plans(parent, op):
    children = build_children(parent)
    total = 1
    for u in range(len(parent)):
        if children[u]:
            total *= len(children[u]) + (1 if op[u] == 0 else 0)
    return total


for n_ in [7, 15, 31, 63, 127]:
    par = [-1] + [(i - 1) // 2 for i in range(1, n_)]       # cây nhị phân đầy đủ
    print(f"n = {n_:>3}: {count_plans(par, [0] * n_):,} kế hoạch")

n =   7: 27 kế hoạch
n =  15: 2,187 kế hoạch
n =  31: 14,348,907 kế hoạch
n =  63: 617,673,396,283,947 kế hoạch
n = 127: 1,144,561,273,430,837,494,885,949,696,427 kế hoạch


Với `n = 127` đã vượt `10^30` kế hoạch. Cần một cách thông minh hơn.

---

# 5. Bước đột phá: nhìn từ dưới lên, mỗi cây con là một bài toán nhỏ

Đặt câu hỏi con cho mỗi đỉnh `u`:

> `dfs(u)` = tổng điểm **lớn nhất** giữ lại được trong cây con của `u`, **giả sử không có tổ tiên nào của `u` xoá cả cây con này**.

Các quan sát then chốt:

1. **Các cây con của những con khác nhau là độc lập.** Lựa chọn bên trong cây con của `c1` không ảnh hưởng gì tới cây con của `c2`. Vậy nếu không xoá con nào, tổng tốt nhất là `score[u] + Σ dfs(c)`.
2. **Xoá con `c` làm mất đúng `dfs(c)` điểm.** Toàn bộ cây con của `c` về 0, nên phần đóng góp của nó từ `dfs(c)` (giá trị tốt nhất có thể) trở thành `0`. Các thao tác bên trong cây con của `c` vẫn phải "thực hiện" nhưng không còn tác dụng gì nữa — không gây xung đột.
3. Muốn **mất ít nhất**, ta xoá con có `dfs(c)` **nhỏ nhất**. Nếu `dfs(c)` âm thì xoá còn là **được lợi**.

Gọi:

```text
tot = score[u] + Σ dfs(c)          (tổng nếu không xoá gì)
mn  = min dfs(c)                   (con "tệ nhất")
```

Công thức:

| Trường hợp | `dfs(u)` | Giải thích |
|---|---|---|
| `u` là lá | `score[u]` | không có thao tác |
| `op[u] = 1` | `tot - mn` | **bắt buộc** xoá một con → xoá con tệ nhất |
| `op[u] = 0` | `tot - min(0, mn)` | chỉ xoá khi có lợi (`mn < 0`) |

Đáp án: `dfs(0)`.

## Tính tay với ví dụ 1

| Đỉnh | op | Các con và `dfs(con)` | tot | mn | dfs(u) |
|---:|:---:|---|---:|---:|---:|
| 4, 5, 6, 7 | (lá) | — | | | `3, 2, 3, -3` |
| 1 | 1 | 4 → 3 | 6 + 3 = 9 | 3 | `9 - 3 = 6` |
| 2 | 0 | 5 → 2 | 1 + 2 = 3 | 2 | `3 - min(0, 2) = 3` |
| 3 | 0 | 6 → 3, 7 → -3 | 2 + 3 - 3 = 2 | -3 | `2 - (-3) = 5` |
| 0 | 1 | 1 → 6, 2 → 3, 3 → 5 | 5 + 6 + 3 + 5 = 19 | 3 | `19 - 3 = 16` |

`dfs(0) = 16` ✔ — gốc xoá con `2` (có `dfs` nhỏ nhất là `3`).

# 6. Cách 2 — DFS đệ quy (dễ đọc)

Cài đặt đúng nguyên văn công thức. Cách này dễ hiểu nhất, nhưng sẽ bị giới hạn độ sâu đệ quy của Python với cây sâu (xem mục lỗi thường gặp).

In [7]:
def max_score_recursive(parent: List[int], op: List[int], score: List[int]) -> int:
    children = build_children(parent)

    def dfs(u: int) -> int:
        if not children[u]:
            return score[u]                        # lá
        vals = [dfs(c) for c in children[u]]
        tot = score[u] + sum(vals)
        mn = min(vals)
        if op[u] == 1:
            return tot - mn                        # bắt buộc xoá con tệ nhất
        return tot - min(0, mn)                    # chỉ xoá nếu có lợi

    return dfs(0)


print(max_score_recursive([-1, 0, 0, 0, 1, 2, 3, 3], [1, 1, 0, 0, 1, 1, 0, 1], [5, 6, 1, 2, 3, 2, 3, -3]))  # 16
print(max_score_recursive([-1, 0, 0], [0, 0, 0], [1, 2, 3]))   # 6
print(max_score_recursive([-1], [1], [-10]))                   # -10

16
6
-10


# 7. Cách 3 — DP lặp theo thứ tự BFS ngược (an toàn với cây sâu)

Thay đệ quy bằng thứ tự `reversed(order)`: khi xử lý `u`, mọi con của `u` đã có `best[c]`.

Để không phải tạo list `vals` cho mỗi đỉnh, ta tính `tot` và `mn` trực tiếp trong một vòng lặp qua các con.

In [8]:
def max_score_iterative(parent: List[int], op: List[int], score: List[int]) -> int:
    n = len(parent)
    children = build_children(parent)
    order = [0]
    for u in order:
        order.extend(children[u])

    best = [0] * n
    for u in reversed(order):                  # con trước cha
        if not children[u]:
            best[u] = score[u]
            continue
        tot = score[u]
        mn = float("inf")
        for c in children[u]:
            tot += best[c]
            mn = min(mn, best[c])
        best[u] = tot - mn if op[u] == 1 else tot - min(0, mn)
    return best[0]


print(max_score_iterative([-1, 0, 0, 0, 1, 2, 3, 3], [1, 1, 0, 0, 1, 1, 0, 1], [5, 6, 1, 2, 3, 2, 3, -3]))  # 16
print(max_score_iterative([-1, 0, 0], [0, 0, 0], [1, 2, 3]))   # 6
print(max_score_iterative([-1], [1], [-10]))                   # -10

16
6
-10


In [9]:
def trace_iterative(parent: List[int], op: List[int], score: List[int]) -> int:
    n = len(parent)
    children = build_children(parent)
    order = [0]
    for u in order:
        order.extend(children[u])
    best = [0] * n
    for u in reversed(order):
        if not children[u]:
            best[u] = score[u]
            print(f"đỉnh {u}: lá -> best = score = {best[u]}")
            continue
        vals = {c: best[c] for c in children[u]}
        tot = score[u] + sum(vals.values())
        worst = min(vals, key=vals.get)
        mn = vals[worst]
        if op[u] == 1:
            best[u] = tot - mn
            action = f"op=1: bắt buộc xoá con {worst} (dfs = {mn})"
        elif mn < 0:
            best[u] = tot - mn
            action = f"op=0: xoá con {worst} vì dfs = {mn} < 0"
        else:
            best[u] = tot
            action = "op=0: không xoá gì"
        print(f"đỉnh {u}: con {vals}, tot = {tot}, {action} -> best = {best[u]}")
    print("Đáp án:", best[0])
    return best[0]


trace_iterative([-1, 0, 0, 0, 1, 2, 3, 3], [1, 1, 0, 0, 1, 1, 0, 1], [5, 6, 1, 2, 3, 2, 3, -3]);

đỉnh 7: lá -> best = score = -3
đỉnh 6: lá -> best = score = 3
đỉnh 5: lá -> best = score = 2
đỉnh 4: lá -> best = score = 3
đỉnh 3: con {6: 3, 7: -3}, tot = 2, op=0: xoá con 7 vì dfs = -3 < 0 -> best = 5
đỉnh 2: con {5: 2}, tot = 3, op=0: không xoá gì -> best = 3
đỉnh 1: con {4: 3}, tot = 9, op=1: bắt buộc xoá con 4 (dfs = 3) -> best = 6
đỉnh 0: con {1: 6, 2: 3, 3: 5}, tot = 19, op=1: bắt buộc xoá con 2 (dfs = 3) -> best = 16
Đáp án: 16


# 8. Phiên bản đúng format LeetCode

In [10]:
from typing import List


class Solution:
    def maxScore(self, parent: List[int], op: List[int], score: List[int]) -> int:
        n = len(parent)
        children = [[] for _ in range(n)]
        for i in range(1, n):
            children[parent[i]].append(i)

        order = [0]
        for u in order:
            order.extend(children[u])

        best = [0] * n
        for u in reversed(order):
            if not children[u]:
                best[u] = score[u]
                continue
            tot = score[u]
            mn = float("inf")
            for c in children[u]:
                tot += best[c]
                if best[c] < mn:
                    mn = best[c]
            best[u] = tot - mn if op[u] == 1 else tot - min(0, mn)
        return best[0]

In [11]:
sol = Solution()

tests = [
    ([-1, 0, 0, 0, 1, 2, 3, 3], [1, 1, 0, 0, 1, 1, 0, 1], [5, 6, 1, 2, 3, 2, 3, -3], 16, "ví dụ 1"),
    ([-1, 0, 0], [0, 0, 0], [1, 2, 3], 6, "ví dụ 2"),
    ([-1], [1], [-10], -10, "ví dụ 3: chỉ có lá"),
    ([-1, 0], [1, 0], [1, 5], 1, "op=1 buộc xoá con tốt"),
    ([-1, 0], [0, 0], [1, -5], 1, "op=0 xoá con âm"),
    ([-1, 0], [1, 0], [1, -5], 1, "op=1 xoá con âm (vừa bắt buộc vừa có lợi)"),
    ([-1, 0, 1, 2], [1, 1, 1, 0], [1, 2, 3, 4], 1, "đường thẳng toàn op=1"),
    ([-1, 0, 0, 2], [1, 0, 1, 0], [0, 5, 4, 10], 5, "chọn theo dfs, không theo tổng gốc"),
]

for parent_, op_, score_, expected, name in tests:
    result = sol.maxScore(parent_, op_, score_)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:42} -> {result}")

PASS | ví dụ 1                                    -> 16
PASS | ví dụ 2                                    -> 6
PASS | ví dụ 3: chỉ có lá                         -> -10
PASS | op=1 buộc xoá con tốt                      -> 1
PASS | op=0 xoá con âm                            -> 1
PASS | op=1 xoá con âm (vừa bắt buộc vừa có lợi)  -> 1
PASS | đường thẳng toàn op=1                      -> 1
PASS | chọn theo dfs, không theo tổng gốc         -> 5


Giải thích test "đường thẳng toàn `op = 1`" (`0 — 1 — 2 — 3`): đỉnh `0` bắt buộc xoá con duy nhất `1` → cả cây con `1, 2, 3` về 0 → chỉ còn `score[0] = 1`. Các thao tác của `1` và `2` vẫn "được thực hiện" nhưng không còn tác dụng.

---

# 9. Vì sao thuật toán đúng?

Chứng minh bằng quy nạp từ lá lên: giả sử `dfs(c)` đúng với mọi con `c` của `u`.

- **Mọi kế hoạch** cho cây con của `u` gồm: lựa chọn tại `u` (xoá con `c*` hoặc không), và kế hoạch riêng trong từng cây con.
- Nếu `u` xoá `c*`: cây con `c*` đóng góp `0` bất kể kế hoạch bên trong; các con khác đóng góp tối đa `dfs(c)` mỗi con, và **có thể đạt cùng lúc** vì chúng độc lập → tổng tốt nhất `score[u] + Σ_{c ≠ c*} dfs(c) = tot - dfs(c*)`.
- Nếu `u` không xoá gì (chỉ được khi `op[u] = 0`): tổng tốt nhất `tot`.
- Lấy max trên các lựa chọn hợp lệ:
  - `op = 1`: `max_{c*} (tot - dfs(c*)) = tot - min dfs = tot - mn`;
  - `op = 0`: `max(tot, tot - mn) = tot - min(0, mn)`.

Giả thiết "không tổ tiên nào xoá cây con của `u`" chính là trường hợp `u` (và cả cây con) còn ý nghĩa; nếu tổ tiên xoá, cây con đóng góp `0` và đã được tính ở cấp trên.

---

## Trực giác

> Mỗi đỉnh là một "quản lý" nhận báo cáo từ các nhóm con: *"nhóm tôi giữ được tối đa X điểm"*. Nếu buộc phải giải tán một nhóm, quản lý giải tán nhóm báo cáo **thấp nhất**. Nếu được quyền không giải tán, chỉ giải tán khi nhóm đó đang **lỗ**.

# 10. Độ phức tạp

Gọi `n` là số đỉnh.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---:|---:|
| 1. Brute Force | thử mọi kế hoạch | tích số lựa chọn × `O(n)` (mũ) | `O(n)` |
| 2. DFS đệ quy | công thức `dfs(u)` | `O(n)` | `O(n)` (stack đệ quy — có thể vượt giới hạn) |
| 3. DP lặp | `reversed(order)` | `O(n)` | `O(n)` |

Mỗi đỉnh được xử lý một lần, mỗi cạnh cha–con được duyệt một lần → tổng `O(n)`.

# 11. Đối chiếu các cách trên cây ngẫu nhiên nhỏ

In [12]:
import random

random.seed(4112)
for _ in range(2000):
    n_ = random.randint(1, 8)
    parent_ = [-1] + [random.randrange(i) for i in range(1, n_)]
    op_ = [random.randint(0, 1) for _ in range(n_)]
    score_ = [random.randint(-5, 5) for _ in range(n_)]
    expected = max_score_bruteforce(parent_, op_, score_)
    assert max_score_recursive(parent_, op_, score_) == expected
    assert max_score_iterative(parent_, op_, score_) == expected
    assert sol.maxScore(parent_, op_, score_) == expected

print("2000 cây ngẫu nhiên: 4 hàm cho cùng kết quả!")

2000 cây ngẫu nhiên: 4 hàm cho cùng kết quả!


Và trên cây lớn — một đường thẳng sâu `10^5` đỉnh:

In [13]:
import time

N = 10**5
chain_parent = [-1] + list(range(N - 1))            # 0 - 1 - 2 - ... - (N-1)
chain_op = [random.randint(0, 1) for _ in range(N)]
chain_score = [random.randint(-10**4, 10**4) for _ in range(N)]

t0 = time.perf_counter()
print("DP lặp:", sol.maxScore(chain_parent, chain_op, chain_score), f"({time.perf_counter() - t0:.2f}s)")

try:
    max_score_recursive(chain_parent, chain_op, chain_score)
except RecursionError as e:
    print("DFS đệ quy: RecursionError -", str(e)[:45])

DP lặp: 8688 (0.04s)


DFS đệ quy: RecursionError - maximum recursion depth exceeded


# 12. Những lỗi dễ mắc

## Lỗi 1 — Bẫy `children[-1]` khi dựng danh sách con

Nếu duyệt từ `i = 0`, ta sẽ chạy `children[parent[0]]` tức `children[-1]` — Python **không báo lỗi** mà hiểu là **phần tử cuối** của list! Gốc `0` bị gắn làm con của đỉnh `n - 1` → cây có chu trình, BFS chạy vô hạn hoặc kết quả sai.

In [14]:
parent_demo = [-1, 0, 0]
bad_children = [[] for _ in range(3)]
for i in range(3):                                 # SAI: phải bắt đầu từ 1
    bad_children[parent_demo[i]].append(i)
print("Sai :", bad_children, "<- đỉnh 2 'có con' là gốc 0!")
print("Đúng:", build_children(parent_demo))

Sai : [[1, 2], [], [0]] <- đỉnh 2 'có con' là gốc 0!
Đúng: [[1, 2], [], []]


## Lỗi 2 — Coi `op = 1` là tùy chọn

Dùng `tot - min(0, mn)` cho cả hai loại `op` sẽ bỏ qua nghĩa vụ "bắt buộc xoá đúng một con".

In [15]:
def wrong_op1_optional(parent, op, score):
    n = len(parent)
    children = build_children(parent)
    order = [0]
    for u in order:
        order.extend(children[u])
    best = [0] * n
    for u in reversed(order):
        if not children[u]:
            best[u] = score[u]
            continue
        vals = [best[c] for c in children[u]]
        best[u] = score[u] + sum(vals) - min(0, min(vals))      # SAI với op = 1
    return best[0]


print("Sai :", wrong_op1_optional([-1, 0], [1, 0], [1, 5]))
print("Đúng:", sol.maxScore([-1, 0], [1, 0], [1, 5]))

Sai : 6
Đúng: 1


## Lỗi 3 — Chọn con để xoá theo **tổng điểm ban đầu** của cây con

Số điểm mất khi xoá con `c` là `dfs(c)` — phần **còn lại sau khi** các thao tác bên trong cây con `c` đã được tối ưu — chứ không phải tổng điểm ban đầu `subtree_sum[c]`.

Ví dụ: gốc (`op = 1`, điểm 0) có hai con: lá `1` (điểm 5) và đỉnh `2` (`op = 1`, điểm 4) có một lá con `3` (điểm 10). Đỉnh `2` bắt buộc xoá lá `3`, nên `dfs(2) = 4`, dù `subtree_sum[2] = 14`.

In [16]:
def wrong_choose_by_original_sum(parent, op, score):
    n = len(parent)
    children = build_children(parent)
    order = [0]
    for u in order:
        order.extend(children[u])
    best = [0] * n
    subtree_sum = [0] * n
    for u in reversed(order):
        subtree_sum[u] = score[u] + sum(subtree_sum[c] for c in children[u])
        if not children[u]:
            best[u] = score[u]
            continue
        tot = score[u] + sum(best[c] for c in children[u])
        worst = min(children[u], key=lambda c: subtree_sum[c])     # SAI: phải theo best[c]
        if op[u] == 1 or best[worst] < 0:
            best[u] = tot - best[worst]
        else:
            best[u] = tot
    return best[0]


p, o, s = [-1, 0, 0, 2], [1, 0, 1, 0], [0, 5, 4, 10]
print("Sai :", wrong_choose_by_original_sum(p, o, s))
print("Đúng:", sol.maxScore(p, o, s))

Sai : 4
Đúng: 5


Code sai xoá lá `1` (vì `subtree_sum[1] = 5 < 14`) và chỉ giữ được `4`. Đúng ra nên xoá con `2` (vì `dfs(2) = 4 < 5`) và giữ được `5`.

## Lỗi 4 — Áp dụng thao tác cho lá

Lá không có con nên không có thao tác, dù `op` của nó bằng `1`. Nếu code cố tìm `min` trên danh sách con rỗng sẽ gặp lỗi `ValueError: min() arg is an empty sequence`. Ví dụ 3 bắt lỗi này.

## Lỗi 5 — Dùng đệ quy cho cây sâu

Như mục 11: đường thẳng `10^5` đỉnh làm DFS đệ quy báo `RecursionError`. Dùng thứ tự BFS ngược.

## Lỗi 6 — Nghĩ rằng phải mô phỏng theo thứ tự thao tác

Thứ tự không quan trọng (mục 3). Cố mô phỏng "ai làm trước" chỉ làm bài toán rối và chậm thêm.

# 13. Mẹo nhận diện

1. Cây có gốc, mỗi đỉnh đưa ra **quyết định về các con** (chọn / bỏ / xoá một nhánh) → DP trả về **giá trị tốt nhất của mỗi cây con**.
2. Nếu quyết định ở đỉnh `u` chỉ ảnh hưởng **bên trong cây con của `u`** và các cây con anh em độc lập → công thức dạng `score[u] + Σ dfs(c)` rồi điều chỉnh theo lựa chọn.
3. "Bắt buộc chọn đúng một" → trừ `min` (hoặc cộng `max`). "Chọn nhiều nhất một" → trừ `min(0, min)`.
4. Cây cho bằng mảng `parent` → dựng `children` (bắt đầu từ `i = 1`), duyệt BFS rồi xử lý ngược.

Các bài cùng họ: *House Robber III* (LeetCode 337), *Maximum Score After Applying Operations on a Tree* (2925), *Binary Tree Maximum Path Sum* (124), và bài 4108 trong cùng bộ notebook này.

# 14. Biến thể

## Biến thể 1 — Mỗi đỉnh được xoá **nhiều nhất `t[u]`** con

Sắp các giá trị `dfs(c)` tăng dần, xoá các con có `dfs` **âm** nhỏ nhất, tối đa `t[u]` con.

In [17]:
def max_score_at_most_t(parent: List[int], t: List[int], score: List[int]) -> int:
    n = len(parent)
    children = build_children(parent)
    order = [0]
    for u in order:
        order.extend(children[u])
    best = [0] * n
    for u in reversed(order):
        vals = sorted(best[c] for c in children[u])
        removed = sum(v for v in vals[:t[u]] if v < 0)      # chỉ xoá các nhánh âm
        best[u] = score[u] + sum(vals) - removed
    return best[0]


print(max_score_at_most_t([-1, 0, 0, 0], [2, 0, 0, 0], [1, -3, -4, 5]))   # xoá cả -3 và -4: 6
print(max_score_at_most_t([-1, 0, 0, 0], [1, 0, 0, 0], [1, -3, -4, 5]))   # chỉ xoá -4: 3

6
3


## Biến thể 2 — Tìm tổng điểm **nhỏ nhất**

Đối xứng: `op = 1` bắt buộc xoá con có `dfs` **lớn nhất**; `op = 0` chỉ xoá khi `max > 0`.

In [18]:
def min_score(parent: List[int], op: List[int], score: List[int]) -> int:
    n = len(parent)
    children = build_children(parent)
    order = [0]
    for u in order:
        order.extend(children[u])
    best = [0] * n
    for u in reversed(order):
        if not children[u]:
            best[u] = score[u]
            continue
        vals = [best[c] for c in children[u]]
        tot = score[u] + sum(vals)
        mx = max(vals)
        best[u] = tot - mx if op[u] == 1 else tot - max(0, mx)
    return best[0]


print(min_score([-1, 0, 0, 0, 1, 2, 3, 3], [1, 1, 0, 0, 1, 1, 0, 1], [5, 6, 1, 2, 3, 2, 3, -3]))

5


# 15. Template tư duy

```python
children = [[] for _ in range(n)]
for i in range(1, n):                     # bỏ gốc
    children[parent[i]].append(i)

order = [0]
for u in order:
    order.extend(children[u])

best = [0] * n
for u in reversed(order):                 # con trước cha
    if not children[u]:
        best[u] = <giá trị của lá>
        continue
    tot = score[u] + sum(best[c] for c in children[u])
    mn = min(best[c] for c in children[u])
    best[u] = tot - mn            # bắt buộc chọn một
    # best[u] = tot - min(0, mn)  # chọn nhiều nhất một
answer = best[0]
```

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

```python
parent = [-1, 0, 0, 1, 1, 2]
op     = [ 0, 1, 1, 0, 0, 0]
score  = [ 2, -1, 4, 6, -2, 3]
```

1. Vẽ cây.
2. Tính `dfs` của từng đỉnh từ lá lên gốc (lập bảng như mục 5).
3. Đáp án là bao nhiêu?

Kiểm tra bằng cell dưới.

In [19]:
# Kiểm tra Bài 1
trace_iterative([-1, 0, 0, 1, 1, 2], [0, 1, 1, 0, 0, 0], [2, -1, 4, 6, -2, 3]);

đỉnh 5: lá -> best = score = 3
đỉnh 4: lá -> best = score = -2
đỉnh 3: lá -> best = score = 6
đỉnh 2: con {5: 3}, tot = 7, op=1: bắt buộc xoá con 5 (dfs = 3) -> best = 4
đỉnh 1: con {3: 6, 4: -2}, tot = 3, op=1: bắt buộc xoá con 4 (dfs = -2) -> best = 5
đỉnh 0: con {1: 5, 2: 4}, tot = 11, op=0: không xoá gì -> best = 11
Đáp án: 11


## Bài 2 — Dự đoán output

```python
ch = build_children([-1, 2, 0, 2, 1])
order = [0]
for u in order:
    order.extend(ch[u])
print(ch)
print(order)
```

In [20]:
ch = build_children([-1, 2, 0, 2, 1])
order_demo = [0]
for u in order_demo:
    order_demo.extend(ch[u])
print(ch)
print(order_demo)

[[2], [4], [1, 3], [], []]
[0, 2, 1, 3, 4]


## Bài 3 — Hoàn thiện DP lặp

In [21]:
def exercise_3_max_score(parent: List[int], op: List[int], score: List[int]) -> int:
    n = len(parent)
    children = build_children(parent)
    order = [0]
    for u in order:
        order.extend(children[u])
    best = [0] * n
    for u in reversed(order):
        # TODO 1: nếu u là lá -> best[u] = score[u]
        # TODO 2: tính tot và mn từ các con
        # TODO 3: op = 1 -> tot - mn ; op = 0 -> tot - min(0, mn)
        pass
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_max_score([-1, 0, 0, 0, 1, 2, 3, 3], [1, 1, 0, 0, 1, 1, 0, 1], [5, 6, 1, 2, 3, 2, 3, -3]) == 16
# assert exercise_3_max_score([-1], [1], [-10]) == -10

## Bài 4 — Độ sâu lớn nhất của cây (không đệ quy)

Viết hàm trả về độ sâu lớn nhất (gốc có độ sâu `0`) của cây cho bằng mảng `parent`, chạy được với `n = 10^5`.

```python
max_depth([-1, 0, 0, 0, 1, 2, 3, 3])   -> 2
max_depth([-1, 0, 1, 2])               -> 3
```

In [22]:
def exercise_4_max_depth(parent: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_max_depth([-1, 0, 0, 0, 1, 2, 3, 3]) == 2
# assert exercise_4_max_depth([-1, 0, 1, 2]) == 3

## Bài 5 — Biến thể: thêm loại `op = 2`

Thêm loại thao tác `op[u] = 2`: **bắt buộc xoá đúng hai con** (giả sử đỉnh loại này luôn có ít nhất 2 con). Sửa thuật toán cho cả ba loại.

```python
max_score_with_op2([-1, 0, 0, 0], [2, 0, 0, 0], [1, 3, 4, 5])   -> 6     # xoá hai con 3 và 4
max_score_with_op2([-1, 0, 0, 0], [2, 0, 0, 0], [1, -3, 4, 5])  -> 6     # xoá -3 và 4
```

In [23]:
def exercise_5_max_score_with_op2(parent: List[int], op: List[int], score: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_max_score_with_op2([-1, 0, 0, 0], [2, 0, 0, 0], [1, 3, 4, 5]) == 6
# assert exercise_5_max_score_with_op2([-1, 0, 0, 0], [2, 0, 0, 0], [1, -3, 4, 5]) == 6

## Bài 6 — Mini challenge: chỉ ra kế hoạch tối ưu

Trả về `(điểm tối đa, plan)` với `plan[u]` là con mà đỉnh `u` xoá (hoặc `-1` nếu không xoá / `u` là lá). Kiểm tra bằng cách **mô phỏng** kế hoạch và so tổng điểm.

```python
max_score_with_plan([-1, 0, 0, 0, 1, 2, 3, 3], [1, 1, 0, 0, 1, 1, 0, 1], [5, 6, 1, 2, 3, 2, 3, -3])
-> (16, [2, 4, -1, 7, -1, -1, -1, -1])
```

In [24]:
def exercise_6_max_score_with_plan(parent: List[int], op: List[int], score: List[int]):
    # TODO
    raise NotImplementedError


# value, plan = exercise_6_max_score_with_plan([-1, 0, 0, 0, 1, 2, 3, 3], [1, 1, 0, 0, 1, 1, 0, 1], [5, 6, 1, 2, 3, 2, 3, -3])
# assert value == 16 and plan == [2, 4, -1, 7, -1, -1, -1, -1]

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `if not children[u]: best[u] = score[u]; continue`.
- Mức 2: `tot = score[u] + sum(best[c] for c in children[u])`, `mn = min(best[c] for c in children[u])`.
- Mức 3: `best[u] = tot - mn if op[u] == 1 else tot - min(0, mn)`; cuối cùng `return best[0]`.

**Bài 4**

- Mức 1: độ sâu của con = độ sâu của cha + 1.
- Mức 2: duyệt theo thứ tự BFS (cha trước con).
- Mức 3: `depth[c] = depth[u] + 1` cho mỗi con `c` của `u`; trả về `max(depth)`.

**Bài 5**

- Mức 1: sắp các `best[c]` tăng dần thành `vals`.
- Mức 2: `op = 2` → trừ `vals[0] + vals[1]`.
- Mức 3: `op = 1` → trừ `vals[0]`; `op = 0` → trừ `min(0, vals[0])`.

**Bài 6**

- Mức 1: khi tính `best[u]`, nhớ luôn con `worst` đạt `mn`.
- Mức 2: `plan[u] = worst` nếu `op[u] == 1` hoặc `mn < 0`, ngược lại `-1`.
- Mức 3: mô phỏng: đặt cờ `zero` cho cây con của mọi `plan[u] != -1`, rồi cộng điểm các đỉnh không bị xoá.

# 17. Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

## Đáp án Bài 1

```text
            0 (op=0, 2)
           /           \
     1 (op=1, -1)     2 (op=1, 4)
      /      \             |
   3 (6)   4 (-2)        5 (3)
```

| Đỉnh | op | Con → dfs | tot | mn | dfs(u) |
|---:|:---:|---|---:|---:|---:|
| 3, 4, 5 | lá | — | | | `6, -2, 3` |
| 1 | 1 | 3 → 6, 4 → -2 | -1 + 6 - 2 = 3 | -2 | `3 - (-2) = 5` |
| 2 | 1 | 5 → 3 | 4 + 3 = 7 | 3 | `7 - 3 = 4` |
| 0 | 0 | 1 → 5, 2 → 4 | 2 + 5 + 4 = 11 | 4 | `11 - min(0, 4) = 11` |

Đáp án `11`: đỉnh `1` xoá lá âm `4`, đỉnh `2` buộc xoá lá `5`, gốc không xoá gì. Điểm còn lại `2 - 1 + 4 + 6 = 11`.

## Đáp án Bài 2

`parent = [-1, 2, 0, 2, 1]`: con của `0` là `[2]`, con của `2` là `[1, 3]`, con của `1` là `[4]`.

```text
[[2], [4], [1, 3], [], []]
[0, 2, 1, 3, 4]
```

In [25]:
# Đáp án Bài 3
def answer_3_max_score(parent: List[int], op: List[int], score: List[int]) -> int:
    n = len(parent)
    children = build_children(parent)
    order = [0]
    for u in order:
        order.extend(children[u])
    best = [0] * n
    for u in reversed(order):
        if not children[u]:
            best[u] = score[u]
            continue
        tot = score[u] + sum(best[c] for c in children[u])
        mn = min(best[c] for c in children[u])
        best[u] = tot - mn if op[u] == 1 else tot - min(0, mn)
    return best[0]


assert answer_3_max_score([-1, 0, 0, 0, 1, 2, 3, 3], [1, 1, 0, 0, 1, 1, 0, 1], [5, 6, 1, 2, 3, 2, 3, -3]) == 16
assert answer_3_max_score([-1], [1], [-10]) == -10
assert answer_3_max_score([-1, 0, 0, 1, 1, 2], [0, 1, 1, 0, 0, 0], [2, -1, 4, 6, -2, 3]) == 11
print("Bài 3: PASS")

Bài 3: PASS


In [26]:
# Đáp án Bài 4
def answer_4_max_depth(parent: List[int]) -> int:
    n = len(parent)
    children = build_children(parent)
    depth = [0] * n
    order = [0]
    for u in order:
        for c in children[u]:
            depth[c] = depth[u] + 1
        order.extend(children[u])
    return max(depth)


assert answer_4_max_depth([-1, 0, 0, 0, 1, 2, 3, 3]) == 2
assert answer_4_max_depth([-1, 0, 1, 2]) == 3
assert answer_4_max_depth(chain_parent) == N - 1
print("Bài 4: PASS")

Bài 4: PASS


In [27]:
# Đáp án Bài 5
def answer_5_max_score_with_op2(parent: List[int], op: List[int], score: List[int]) -> int:
    n = len(parent)
    children = build_children(parent)
    order = [0]
    for u in order:
        order.extend(children[u])
    best = [0] * n
    for u in reversed(order):
        if not children[u]:
            best[u] = score[u]
            continue
        vals = sorted(best[c] for c in children[u])
        tot = score[u] + sum(vals)
        if op[u] == 2:
            best[u] = tot - vals[0] - vals[1]
        elif op[u] == 1:
            best[u] = tot - vals[0]
        else:
            best[u] = tot - min(0, vals[0])
    return best[0]


assert answer_5_max_score_with_op2([-1, 0, 0, 0], [2, 0, 0, 0], [1, 3, 4, 5]) == 6
assert answer_5_max_score_with_op2([-1, 0, 0, 0], [2, 0, 0, 0], [1, -3, 4, 5]) == 6
assert answer_5_max_score_with_op2([-1, 0, 0, 0, 1, 2, 3, 3], [1, 1, 0, 0, 1, 1, 0, 1], [5, 6, 1, 2, 3, 2, 3, -3]) == 16
print("Bài 5: PASS")

Bài 5: PASS


In [28]:
# Đáp án Bài 6
def answer_6_max_score_with_plan(parent: List[int], op: List[int], score: List[int]):
    n = len(parent)
    children = build_children(parent)
    order = [0]
    for u in order:
        order.extend(children[u])
    best = [0] * n
    plan = [-1] * n
    for u in reversed(order):
        if not children[u]:
            best[u] = score[u]
            continue
        tot = score[u] + sum(best[c] for c in children[u])
        worst = min(children[u], key=lambda c: best[c])
        if op[u] == 1 or best[worst] < 0:
            plan[u] = worst
            best[u] = tot - best[worst]
        else:
            best[u] = tot
    return best[0], plan


def simulate(parent, plan, score):
    children = build_children(parent)
    zero = [False] * len(parent)
    for chosen in plan:
        if chosen == -1:
            continue
        stack = [chosen]
        while stack:
            v = stack.pop()
            zero[v] = True
            stack.extend(children[v])
    return sum(s for v, s in enumerate(score) if not zero[v])


value, plan = answer_6_max_score_with_plan([-1, 0, 0, 0, 1, 2, 3, 3], [1, 1, 0, 0, 1, 1, 0, 1], [5, 6, 1, 2, 3, 2, 3, -3])
print(value, plan)
assert value == 16 and plan == [2, 4, -1, 7, -1, -1, -1, -1]
random.seed(6)
for _ in range(500):
    n_ = random.randint(1, 8)
    parent_ = [-1] + [random.randrange(i) for i in range(1, n_)]
    op_ = [random.randint(0, 1) for _ in range(n_)]
    score_ = [random.randint(-5, 5) for _ in range(n_)]
    v, pl = answer_6_max_score_with_plan(parent_, op_, score_)
    ch_ = build_children(parent_)
    assert all(pl[u] != -1 for u in range(n_) if ch_[u] and op_[u] == 1)   # op=1 luôn xoá một con
    assert simulate(parent_, pl, score_) == v == max_score_bruteforce(parent_, op_, score_)
print("Bài 6: PASS (kế hoạch hợp lệ, mô phỏng đúng điểm, khớp vét cạn)")

16 [2, 4, -1, 7, -1, -1, -1, -1]
Bài 6: PASS (kế hoạch hợp lệ, mô phỏng đúng điểm, khớp vét cạn)


# 18. Tổng kết

Bài **Maximize Score After Zeroing Subtrees** là ví dụ mẫu mực của tree DP:

1. **Thứ tự thao tác không quan trọng** → chỉ cần quan tâm tập lựa chọn.
2. **Câu hỏi con**: `dfs(u)` = điểm giữ lại tối đa trong cây con `u`.
3. **Các cây con độc lập** → `tot = score[u] + Σ dfs(c)`; xoá con `c` mất đúng `dfs(c)`.
4. `op = 1` → trừ `min`; `op = 0` → trừ `min(0, min)`.
5. Cài đặt lặp theo **BFS ngược** để an toàn với cây sâu.

## Công thức nhớ nhanh

```text
lá:     dfs(u) = score[u]
op = 1: dfs(u) = tot - mn
op = 0: dfs(u) = tot - min(0, mn)
tot = score[u] + Σ dfs(c),  mn = min dfs(c)
```

## Phiên bản nên nhớ (phần lõi)

```python
for u in reversed(order):
    if not children[u]:
        best[u] = score[u]
        continue
    tot = score[u] + sum(best[c] for c in children[u])
    mn = min(best[c] for c in children[u])
    best[u] = tot - mn if op[u] == 1 else tot - min(0, mn)
return best[0]
```

## Độ phức tạp

```text
Time:  O(n)
Space: O(n)
```

# 19. Checklist trước khi submit

- [ ] Dựng `children` bằng vòng lặp **từ `i = 1`** (tránh `children[-1]`)?
- [ ] Lá **không** có thao tác, dù `op` của lá bằng `1`?
- [ ] `op = 1` **bắt buộc** trừ `mn` (kể cả khi `mn` dương)?
- [ ] `op = 0` chỉ trừ khi `mn < 0`: `tot - min(0, mn)`?
- [ ] Chọn con để xoá theo `dfs(c)` (sau tối ưu), không theo tổng điểm ban đầu?
- [ ] Duyệt **không đệ quy** (BFS rồi `reversed(order)`) cho cây sâu `10^5`?
- [ ] Đã test: một đỉnh, cây đường thẳng, điểm âm với `op = 0` và `op = 1`?